# 02N — Therapy-relevant regions


In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu, kruskal, spearmanr, ks_2samp, entropy, norm
from IPython.display import display

PROCESSED = Path('../../data/processed')
df = pd.read_csv(PROCESSED / 'DMD_variants_annotated.csv')

print('Loaded:', PROCESSED / 'DMD_variants_annotated.csv')
print('Shape:', df.shape)


Loaded: ../../data/processed/DMD_variants_annotated.csv
Shape: (11308, 30)


In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for rel in ('../..', '..', '.'):
    cand = (Path.cwd() / rel).resolve()
    if (cand / 'src' / 'utils.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('project root with src/utils.py not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    add_consistency_flags,
    bh_adjust,
    chi2_table,
    ecdf_xy,
    fisher_bool,
    frame_mismatch,
    kruskal_group,
    mann_whitney_bool,
    mut_cons_mismatch,
    odds_ratio_ci,
    path_cons_mismatch,
    prepare_eda_dataframe,
    spearman_xy,
)

d = prepare_eda_dataframe(df)

print('Prepared shape:', d.shape)


Prepared shape: (11308, 59)


Literal operationalization: the `skip45/51/53` block is interpreted as exon-target status (`exon==45/51/53`), rather than as clinical amenability logic for specific multi-exon deletion patterns.


## 1. 📊 skip51 vs phenotype ⭐


In [4]:
tmp = d[['skip51_region', 'phenotype']].dropna()
fig = px.histogram(tmp, x='skip51_region', color='phenotype', barmode='stack', title='skip51 region vs phenotype')
fig.show()


## 2. 🧪 Fisher


In [5]:
tmp = d[d['phenotype'].isin(['DMD', 'BMD'])][['skip51_region', 'phenotype']].dropna().copy()
tmp['is_dmd'] = tmp['phenotype'].eq('DMD')
tab, odds, p, ci = fisher_bool(tmp, 'skip51_region', 'is_dmd')
display(tab)
display(pd.Series({'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]}).to_frame('value'))


is_dmd,False,True
skip51_region,,
False,1011,7650
True,12,157


,value
odds_ratio,1.729052
p_value,0.068427
or_ci_low,0.957808
or_ci_high,3.121315


## 3. 📊 skip53 vs phenotype


In [6]:
tmp = d[['skip53_region', 'phenotype']].dropna()
fig = px.histogram(tmp, x='skip53_region', color='phenotype', barmode='stack', title='skip53 region vs phenotype')
fig.show()


## 4. 📊 skip45 vs phenotype


In [7]:
tmp = d[['skip45_region', 'phenotype']].dropna()
fig = px.histogram(tmp, x='skip45_region', color='phenotype', barmode='stack', title='skip45 region vs phenotype')
fig.show()


## 5. 📋 comparison table


In [8]:
rows = []
for label, col in [('skip45', 'skip45_region'), ('skip51', 'skip51_region'), ('skip53', 'skip53_region')]:
    tmp = d.copy()
    grp = tmp.groupby(col).agg(
        n=('var_id', 'size'),
        pathogenic_fraction=('pathogenic', 'mean'),
        dmd_fraction=('phenotype', lambda s: (s == 'DMD').mean()),
        bmd_fraction=('phenotype', lambda s: (s == 'BMD').mean())
    ).reset_index().rename(columns={col: 'in_region'})
    grp['region'] = label
    rows.append(grp)
res = pd.concat(rows, ignore_index=True)
display(res)


,in_region,n,pathogenic_fraction,dmd_fraction,bmd_fraction,region
0,False,11130,0.301977,0.691644,0.089308,skip45
1,True,178,0.320225,0.612360,0.162921,skip45
2,False,11084,0.299982,0.690184,0.091213,skip51
3,True,224,0.415179,0.700893,0.053571,skip51
4,False,11139,0.300386,0.689559,0.090403,skip53
5,True,169,0.426036,0.745562,0.094675,skip53
